# Exploratory Data Analysis

Performing Esports Analytics on League of Legends matches. The data contains the korean challanger players ranked matches.

**Origin of the data:** https://developer.riotgames.com

# Importing libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import requests
import json
import matplotlib.ticker as mtick

# Importing the data

In [ ]:
eda_df = pd.read_csv("event_driven_eda_sample.csv")

# Initial data exploration

In [ ]:
eda_df.head()

In [ ]:
eda_df.tail()

In [ ]:
eda_df.shape # Shape of the dataset (rows, columns)

704 features, 2.6 million rows

In [ ]:
eda_df.info() # Summary of the dataset, including data types and non-null counts

In [ ]:
eda_df.columns # List of column names in the dataset

## Explanation of the columns:
* 'match_id': Unique identifier for each match.
* 'minute': The minute of the match when the event occurred.
* 'timestamp': The exact time when the event occurred.
* '*_p(1-10)': Player-specific statistics for each of the 10 players in the match, such as kills, deaths, assists, gold earned, etc.
* 'cum*_*': Cumulative statistics for each team up to that minute, such as cumulative kills, deaths, assists, gold earned, etc.
* '*diff': The difference in a specific statistic between the two teams at that minute, such as gold difference, kill difference, etc.
* 'target_win_t100': Target variable indicating whether team won the game


In [ ]:
eda_df['matchId'].unique() # Unique values in the dataset

2475 unique matches

In [ ]:
eda_df.describe() # Statistical summary of numerical columns

In [ ]:
eda_df[eda_df.duplicated()] # Check for duplicate rows in the dataset

No duplicates.

## Visualizations

In [ ]:
# Get the maximum minute for each match
match_lengths = eda_df.groupby('matchId')['minute'].max()

plt.figure(figsize=(10, 6))
sns.histplot(match_lengths, bins=30, color='#9b59b6', kde=True)

plt.title('Distribution of Match Lengths (Minutes)', fontsize=16, fontweight='bold')
plt.xlabel('Match Length (Minutes)', fontsize=12)
plt.ylabel('Number of Matches', fontsize=12)
plt.axvline(match_lengths.mean(), color='red', linestyle='dashed', linewidth=2, label=f'Mean: {match_lengths.mean():.1f} min')
plt.legend()
plt.show()

In [ ]:
# Get the total kills at the very last minute of each match
# We add Blue (t100) and Red (t200) cumulative kills together
last_minute_df = eda_df.loc[eda_df.groupby('matchId')['minute'].idxmax()]

total_kills = last_minute_df['cum_CHAMPION_KILL_t100'] + last_minute_df['cum_CHAMPION_KILL_t200']
    
plt.figure(figsize=(10, 6))
sns.histplot(total_kills, bins=30, color='#e67e22', kde=True)
    
plt.title('Distribution of Total Kills per Match', fontsize=16, fontweight='bold')
plt.xlabel('Total Kills (Blue + Red)', fontsize=12)
plt.ylabel('Number of Matches', fontsize=12)
plt.axvline(total_kills.mean(), color='red', linestyle='dashed', linewidth=2, label=f'Mean: {total_kills.mean():.1f} kills')
plt.legend()
plt.show()

In [ ]:
# Filter the dataset for the 15th minute
min15_df = eda_df[eda_df['minute'] == 15].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Gold difference distribution
sns.histplot(min15_df['totalGold_diff'], bins=50, ax=axes[0], color='#f1c40f', kde=True)
axes[0].set_title('Gold Difference Distribution (15th Minute)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Gold Difference (Blue - Red)')
axes[0].set_ylabel('Number of Matches')
axes[0].axvline(0, color='black', linestyle='--')

# 2. XP difference distribution
sns.histplot(min15_df['xp_diff'], bins=50, ax=axes[1], color='#2ecc71', kde=True)
axes[1].set_title('XP Difference Distribution (15th Minute)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Experience (XP) Difference (Blue - Red)')
axes[1].set_ylabel('')
axes[1].axvline(0, color='black', linestyle='--')

# 3. Damage difference distribution
sns.histplot(min15_df['dmg_totalDamageDoneToChampions_diff'], bins=50, ax=axes[2], color='#e74c3c', kde=True)
axes[2].set_title('Damage Difference Distribution (15th Minute)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Damage Difference (Blue - Red)')
axes[2].set_ylabel('')
axes[2].axvline(0, color='black', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Most likely to be correlated with the final outcome (win/loss) at the 15th minute:
cols_for_corr = [
    'totalGold_diff', 
    'xp_diff', 
    'level_diff', 
    'cum_CHAMPION_KILL_diff',
    'cum_WARD_PLACED_diff',
    'cum_ELITE_MONSTER_KILL_diff',
    'cum_BUILDING_KILL_diff',
    'dmg_totalDamageDoneToChampions_diff'
]

# Validate that the columns exist in the DataFrame before calculating the correlation matrix
valid_cols = [c for c in cols_for_corr if c in min15_df.columns]

# Calculate the correlation matrix for the valid columns
corr_matrix = min15_df[valid_cols].corr()

# Vizualization of the correlation matrix using a heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, 
            annot=True,          
            cmap='coolwarm',     
            fmt=".2f",           
            vmin=-1, vmax=1,    
            linewidths=0.5, 
            cbar_kws={"shrink": .8})

plt.title('Correlation between Variables at the 15th Minute', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Heatmap for kill coordinates

In [ ]:
# Filter out only CHAMPION_KILL events with valid coordinates
kills_df = eda_df[(eda_df['type'] == 'CHAMPION_KILL') & (eda_df['positionX'].notna())]

# Separate into Early Game (Laning Phase & Early Ganks) and Mid Game (Objective Fights)
early_kills = kills_df[kills_df['minute'] <= 14]
mid_kills = kills_df[(kills_df['minute'] > 14) & (kills_df['minute'] <= 25)]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor('#1a1a1a')

# Subplot 1: Early Game Heatmap (0-14 mins)
axes[0].set_facecolor('#1a1a1a')
sns.kdeplot(x=early_kills['positionX'], y=early_kills['positionY'], 
            cmap="mako", fill=True, thresh=0.05, levels=50, ax=axes[0])
axes[0].set_title("Early Game Kills (0-14 mins)\nFocus: Lanes", color='white', fontsize=16)
axes[0].set_xlim(0, 15000)
axes[0].set_ylim(0, 15000)
axes[0].axis('off')

# Subplot 2: Mid Game Heatmap (15-25 mins)
axes[1].set_facecolor('#1a1a1a')
sns.kdeplot(x=mid_kills['positionX'], y=mid_kills['positionY'], 
            cmap="rocket", fill=True, thresh=0.05, levels=50, ax=axes[1])
axes[1].set_title("Mid Game Kills (15-25 mins)\nFocus: Objectives", color='white', fontsize=16)
axes[1].set_xlim(0, 15000)
axes[1].set_ylim(0, 15000)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Gold and XP difference boxplots for Early vs Mid Game

In [ ]:
# Filter for the Early/Mid game
early_game_df = eda_df[eda_df['minute'] <= 20]

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# 1. Gold Difference Boxplot
sns.boxplot(data=early_game_df, x='minute', y='totalGold_diff', ax=axes[0], color='#f1c40f', showfliers=False)
axes[0].set_title('Early Game Gold Difference (Blue - Red) over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Gold Difference')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.7)

# 2. XP Difference Boxplot
sns.boxplot(data=early_game_df, x='minute', y='xp_diff', ax=axes[1], color='#3498db', showfliers=False)
axes[1].set_title('Early Game XP Difference (Blue - Red) over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('XP Difference')
axes[1].set_xlabel('Minute of the Match')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Macro & Objective Map with Dynamic Marker Sizes

In [ ]:
# Filter out all relevant Objective events (where coordinates are available)
objective_types = ['ELITE_MONSTER_KILL', 'BUILDING_KILL', 'TURRET_PLATE_DESTROYED']
macro_df = eda_df[(eda_df['type'].isin(objective_types)) & (eda_df['positionX'].notna())].copy()

# Set the background color for the entire Figure
fig = plt.figure(figsize=(12, 12), facecolor='#1a1a1a')
ax = plt.gca()
ax.set_facecolor('#1a1a1a')

# ELITE MONSTERS
dragons = macro_df[(macro_df['type'] == 'ELITE_MONSTER_KILL') & (macro_df['monsterType'] == 'DRAGON')]
heralds = macro_df[(macro_df['type'] == 'ELITE_MONSTER_KILL') & (macro_df['monsterType'].isin(['RIFTHERALD', 'BARON_NASHOR', 'HORDE']))]

plt.scatter(dragons['positionX'], dragons['positionY'], 
            color='#e74c3c', marker='o', s=70, alpha=0.5, edgecolors='black', label='Dragons')

plt.scatter(heralds['positionX'], heralds['positionY'], 
            color='#9b59b6', marker='o', s=70, alpha=0.5, edgecolors='black', label='Baron / Herald / Grubs')


# Buildings (Frequency-based size)
buildings = macro_df[macro_df['type'] == 'BUILDING_KILL']

# Group by exact coordinates and count occurrences
tower_counts = buildings.groupby(['positionX', 'positionY']).size().reset_index(name='count')

# Calculate dynamic size: Min size 50, Max size 600 based on relative frequency
tower_sizes = 50 + (tower_counts['count'] / tower_counts['count'].max()) * 550

plt.scatter(tower_counts['positionX'], tower_counts['positionY'], 
            color='#f1c40f', marker='s', s=tower_sizes, alpha=0.7, edgecolors='white', 
            label='Destroyed Buildings (Size = Frequency)')


# Turret plates (Frequency-based size)
plates = macro_df[macro_df['type'] == 'TURRET_PLATE_DESTROYED']

# Group by exact coordinates and count occurrences
plate_counts = plates.groupby(['positionX', 'positionY']).size().reset_index(name='count')

# Calculate dynamic size: Min size 20, Max size 300 based on relative frequency
plate_sizes = 20 + (plate_counts['count'] / plate_counts['count'].max()) * 280

plt.scatter(plate_counts['positionX'], plate_counts['positionY'], 
            color='#3498db', marker='D', s=plate_sizes, alpha=0.6, edgecolors='black', 
            label='Turret Plates (Size = Frequency)')


plt.title("Macro & Objective Map", color='white', fontsize=18, fontweight='bold', pad=20)
plt.xlim(0, 15000)
plt.ylim(0, 15000)

# Legend
legend = plt.legend(loc='upper right', fontsize=12, frameon=True, facecolor='#2c3e50', edgecolor='white', labelcolor='white')
legend.get_frame().set_alpha(0.8)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

## Most purchased items

In [ ]:
# Get current LoL patch version from Riot's DataDragon API
versions_url = "https://ddragon.leagueoflegends.com/api/versions.json"
versions = requests.get(versions_url).json()
latest_version = versions[0]

# Download the Item data for the latest version
items_url = f"https://ddragon.leagueoflegends.com/cdn/{latest_version}/data/en_US/item.json"
items_data = requests.get(items_url).json()

# Dictionary to map item IDs to item names
item_dict = {}
for item_id, item_info in items_data['data'].items():
    item_dict[float(item_id)] = item_info['name']

item_purchases = eda_df[eda_df['type'] == 'ITEM_PURCHASED']['itemId'].dropna()
top_items_raw = item_purchases.value_counts().head(15)

# Renaming the index of top_items_raw using the item_dict
top_items_named = top_items_raw.rename(index=item_dict)

# Vizualization
plt.figure(figsize=(12, 8))
sns.barplot(x=top_items_named.values, y=top_items_named.index.astype(str), palette="viridis")
plt.title('Top 15 Most Purchased Items', fontsize=16, fontweight='bold')
plt.xlabel('Number of Purchases', fontsize=12)
plt.ylabel('Item Name', fontsize=12)
plt.tight_layout()
plt.show()

## Killing Sprees and Multi-Kills Analysis

In [ ]:
# Filter for champion kills to analyze kill streaks and bounties
kills_df = eda_df[eda_df['type'] == 'CHAMPION_KILL'].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Kill streak length distribution
if 'killStreakLength' in kills_df.columns:
    streak_counts = kills_df['killStreakLength'].value_counts().sort_index().head(8)
    
    sns.barplot(x=streak_counts.index.astype(int), y=streak_counts.values, ax=axes[0], color='#e74c3c')
    axes[0].set_title('Kill Streak Length Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Kill Streak (Number of kills without dying)')
    axes[0].set_ylabel('Count')

# Bounties over time
if 'bounty' in kills_df.columns:
    sns.scatterplot(data=kills_df, x='minute', y='bounty', ax=axes[1], color='#f1c40f', alpha=0.3)
    axes[1].set_title('Bounty Values Over Time', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Minute of the Match')
    axes[1].set_ylabel('Bounty Gold Value')
    
    # 300 gold for a standard kill
    axes[1].axhline(300, color='red', linestyle='--', label='Standard Kill (300g)')
    axes[1].legend()

plt.tight_layout()
plt.show()

## Class balance of the target variable (win/loss)

In [ ]:
# Read Match ID and Target Win columns for class balance check
target_cols = ['matchId', 'target_win_t100']
target_df = pd.read_csv("event_driven_eda_sample.csv", usecols=target_cols)

# Only take unique matches
unique_matches = target_df.drop_duplicates(subset=['matchId'])

# Vizualization
plt.figure(figsize=(8, 6))
ax = sns.countplot(data=unique_matches, x='target_win_t100', palette=['#e74c3c', '#3498db'])

plt.title('Blue Team (t100) Win Rate', fontsize=16, fontweight='bold')
plt.xlabel('Did Blue Team Win? (0 = Red Won, 1 = Blue Won)', fontsize=12)
plt.ylabel('Number of Matches', fontsize=12)

# Perctentages
total = len(unique_matches)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%\n({p.get_height()} matches)'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height() + 10
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.ylim(0, unique_matches['target_win_t100'].value_counts().max() * 1.2)
plt.tight_layout()
plt.show()

# Freeing up memory
del target_df, unique_matches

## Win Probability Analysis at 15th and 25th minute based on Gold Difference

In [ ]:
# Read only necessary columns for win probability analysis
prob_cols = ['matchId', 'minute', 'totalGold_diff', 'target_win_t100']
prob_df = pd.read_csv("event_driven_eda_sample.csv", usecols=prob_cols)

# Only 15th and 25th minute (Filter for the first event in that minute)
min15_df = prob_df[prob_df['minute'] == 15].drop_duplicates(subset=['matchId'], keep='first').copy()
min25_df = prob_df[prob_df['minute'] == 25].drop_duplicates(subset=['matchId'], keep='first').copy()

# Function to calculate win probability based on gold difference
def calculate_win_prob(df, feature, bins=15):
    # Categories
    df['bin'] = pd.qcut(df[feature], q=bins, duplicates='drop')
    # Mean gold difference and win rate for each bin
    grouped = df.groupby('bin').agg({
        feature: 'mean',
        'target_win_t100': 'mean',
        'matchId': 'count'
    }).rename(columns={'target_win_t100': 'win_rate'}).reset_index()
    return grouped

prob15 = calculate_win_prob(min15_df, 'totalGold_diff')
prob25 = calculate_win_prob(min25_df, 'totalGold_diff')

plt.figure(figsize=(12, 7))
sns.lineplot(data=prob15, x='totalGold_diff', y='win_rate', marker='o', linewidth=3, markersize=10, label='Minute 15')
sns.lineplot(data=prob25, x='totalGold_diff', y='win_rate', marker='s', linewidth=3, markersize=10, label='Minute 25', color='#e74c3c')

plt.title('Win Probability Curve based on Gold Difference', fontsize=18, fontweight='bold')
plt.xlabel('Gold Advantage (Blue - Red)', fontsize=14)
plt.ylabel('Probability of Blue Team Winning (0.0 to 1.0)', fontsize=14)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7)
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)


plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

plt.tight_layout()
plt.show()

# Freeing up memory
del prob_df, min15_df, min25_df, prob15, prob25

## Analyzing objective control and its impact on win probability at the 15th minute

In [ ]:
# Get relevant columns for objective control analysis
obj_cols = ['matchId', 'minute', 'cum_ELITE_MONSTER_KILL_diff', 'cum_BUILDING_KILL_diff', 'target_win_t100']
obj_df = pd.read_csv("event_driven_eda_sample.csv", usecols=obj_cols)

# Filter for the 15th minute and drop duplicates to get one row per match
min15_obj = obj_df[obj_df['minute'] == 15].drop_duplicates(subset=['matchId'], keep='first')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dragon/Herald (Elite Monster) Advantage
monster_winrate = min15_obj.groupby('cum_ELITE_MONSTER_KILL_diff')['target_win_t100'].agg(['mean', 'count']).reset_index()
# Only look at categories with at least 20 matches to avoid skewing the graph
monster_winrate = monster_winrate[monster_winrate['count'] > 20]

sns.barplot(data=monster_winrate, x='cum_ELITE_MONSTER_KILL_diff', y='mean', ax=axes[0], palette="coolwarm")
axes[0].set_title('Win Rate by Elite Monster Advantage\n(At 15 Minutes)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Monster Kill Difference (Blue - Red)')
axes[0].set_ylabel('Win Probability')
axes[0].axhline(0.5, color='black', linestyle='--')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Tower Advantage
tower_winrate = min15_obj.groupby('cum_BUILDING_KILL_diff')['target_win_t100'].agg(['mean', 'count']).reset_index()
tower_winrate = tower_winrate[tower_winrate['count'] > 20]

sns.barplot(data=tower_winrate, x='cum_BUILDING_KILL_diff', y='mean', ax=axes[1], palette="magma")
axes[1].set_title('Win Rate by Tower Advantage\n(At 15 Minutes)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Destroyed Tower Difference (Blue - Red)')
axes[1].set_ylabel('Win Probability')
axes[1].axhline(0.5, color='black', linestyle='--')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.show()

# Freeing up memory
del obj_df, min15_obj

## Time Dependent Win Probability Analysis based on Gold Difference

In [ ]:
# Filter for the 5th, 10th, 15th, 20th, 25th, and 30th minutes (taking the first event in that minute for each match)
time_df = eda_df[eda_df['minute'].isin([5, 10, 15, 20, 25, 30])].drop_duplicates(subset=['matchId', 'minute'], keep='first').copy()

# Create 7 gold difference categories (bins) from -4000 to +4000
bins = [-np.inf, -4000, -2000, -500, 500, 2000, 4000, np.inf]
labels = ['<-4k', '-4k to -2k', '-2k to -0.5k', 'Even', '0.5k to 2k', '2k to 4k', '>4k']
time_df['gold_bin'] = pd.cut(time_df['totalGold_diff'], bins=bins, labels=labels)

# Calculate the average win probability for each gold bin at each minute
pivot_prob = time_df.pivot_table(index='minute', columns='gold_bin', values='target_win_t100', aggfunc='mean')

plt.figure(figsize=(12, 6))
sns.heatmap(pivot_prob, annot=True, fmt=".0%", cmap="RdYlBu", center=0.5, cbar_kws={'label': 'Blue Team Win Probability'})
plt.title('Time-Dependent Win Probability by Gold Difference', fontsize=16, fontweight='bold')
plt.xlabel('Gold Advantage (Blue - Red)', fontsize=12)
plt.ylabel('Minute of the Match', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()